# pyaegean Stage C — train the biaffine parser

Trains the joint tagger-parser (GreBerta + Stage B tagging heads + biaffine arc/relation
scorers) on the leakage-clean AGDT dataset with UD-convention trees, then measures the
headline numbers on the UD test folds with the official evaluator.

**Targets (UD Perseus test): UAS ≥ 78.8, LAS ≥ 73.1** — the best published numbers.

Runtime: a GPU runtime (G4 or A100; bf16 auto-detects). Run cells top to bottom;
cell 5 is an optional re-train — read its note before using it.

In [ ]:
# 1 — confirm the GPU
import torch
print(torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 2 — install the package + a fresh clone (the training scripts live in the repo, not the wheel)
!pip install -q "git+https://github.com/ryanpavlicek/pyaegean" torch transformers numpy
!rm -rf pyaegean_repo
!git clone https://github.com/ryanpavlicek/pyaegean.git pyaegean_repo

In [ ]:
# 3 — build the parser dataset (fetches AGDT + UD folds to this VM's cache, ~80 MB)
#     sanity check: the printout should say train_sentences: 31112
!python pyaegean_repo/training/build_parser_dataset.py

In [ ]:
# 4 — train (defaults: 6 epochs, lr 3e-5, batch 32, bf16 auto; selection on dev LAS)
#     watch the per-epoch lines: "epoch N: dev UAS …  LAS …  UPOS …"
!python pyaegean_repo/training/train_parser.py --model bowphs/GreBerta

**5 — decision point (one minute).** Look at cell 4's six `epoch N:` lines.

- If the **last** epoch was *not* the best dev LAS (scores flattened or dipped): **skip cell 5.**
- If dev LAS was still climbing at the last epoch: uncomment and run cell 5 — it retrains
  from scratch with a longer budget and re-selects the best epoch.

Tuning on **dev** is fair game; the **test** folds (cell 6) are measured once, at the end.

In [ ]:
# 5 — OPTIONAL longer re-train: uncomment ONLY if dev LAS was still improving at the last epoch
# !python pyaegean_repo/training/train_parser.py --model bowphs/GreBerta --epochs 9

In [ ]:
# 6 — the headline measurements (UD test folds, official evaluator)
!python pyaegean_repo/training/eval_parser_ud.py \
    --checkpoint pyaegean_repo/training/out/parser/model --treebank perseus --split test \
    --out pyaegean_repo/training/out/parser/ud-perseus-test.json
!python pyaegean_repo/training/eval_parser_ud.py \
    --checkpoint pyaegean_repo/training/out/parser/model --treebank proiel --split test \
    --out pyaegean_repo/training/out/parser/ud-proiel-test.json

In [ ]:
# 7 — save everything to Drive BEFORE the runtime recycles
#     (the checkpoint supersedes Stage B's — same tagging heads included; Stage E exports it)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/pyaegean-stage-c"
!cp -r pyaegean_repo/training/out/parser "/content/drive/MyDrive/pyaegean-stage-c/"
!ls -la "/content/drive/MyDrive/pyaegean-stage-c/parser" "/content/drive/MyDrive/pyaegean-stage-c/parser/model"

## What to bring back

Three small JSONs (download them or grab them from Drive):

- `parser/metrics.json` — the training history + best epoch
- `parser/ud-perseus-test.json` — **the verdict**: UAS ≥ 78.8 and LAS ≥ 73.1 clears every
  published number
- `parser/ud-proiel-test.json` — the honest out-of-domain read

The big checkpoint (`parser/model/`, ~520 MB) stays in Drive — it is the artifact Stage E
exports to ONNX.